In [1]:

import pandas as pd
from pathlib import Path

base = Path("../")

# Load the three-way split manifest
manifest_path = base / "outputs/tables/three_way_train_split_manifest.csv"
manifest = pd.read_csv(manifest_path)

# Load affected model rows
affected_path = base / "outputs/tables/affected_model_rows.csv"
affected = pd.read_csv(affected_path, dtype=str)

print("SPLIT MANIFEST")
print("=" * 50)
print("Columns:", manifest.columns.tolist())
print(manifest.head())
print("\nRows:", len(manifest))

print("\nAFFECTED ROWS")
print("=" * 50)
print("Columns:", affected.columns.tolist())
print(affected.head())
print("\nUnique affected rows:", len(affected))

print("\nFILES")
print("Manifest exists:", manifest_path.exists())
print("Affected rows file exists:", affected_path.exists())


SPLIT MANIFEST
Columns: ['train', 'split']
   train                split
0  10103       internal_train
1  10104  reserved_final_test
2  10105           validation
3  10106  reserved_final_test
4  10115       internal_train

Rows: 3146

AFFECTED ROWS
Columns: ['train', 'date', 'current_station', 'next_station']
   train        date current_station next_station
0  12616  2024-09-02             KMT          BZA
1  12616  2024-09-02             BZA          TEL
2  12616  2024-09-02             TEL          CLX
3  12616  2024-09-02             CLX          OGL
4  12616  2024-09-02             OGL          NLR

Unique affected rows: 183

FILES
Manifest exists: True
Affected rows file exists: True


In [2]:

import pandas as pd
import numpy as np
import time
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base = Path("../")

# Load model data
data = pd.read_csv(
    base / "data/processed/railpulse_model_features.csv"
)

# Load split assignments and affected rows
manifest = pd.read_csv(
    base / "outputs/tables/three_way_train_split_manifest.csv"
)

affected = pd.read_csv(
    base / "outputs/tables/affected_model_rows.csv",
    dtype=str
)

# Standardize join keys
for df in [data, manifest, affected]:
    df["train"] = df["train"].astype(str).str.strip()

data["date"] = pd.to_datetime(data["date"]).dt.strftime("%Y-%m-%d")
affected["date"] = pd.to_datetime(affected["date"]).dt.strftime("%Y-%m-%d")

for col in ["current_station", "next_station"]:
    data[col] = data[col].astype(str).str.strip()
    affected[col] = affected[col].astype(str).str.strip()

# Attach original train-level split
data = data.merge(manifest, on="train", how="inner", validate="many_to_one")

# Flag the 183 affected rows using exact row keys
data = data.merge(
    affected.assign(is_affected=True),
    on=["train", "date", "current_station", "next_station"],
    how="left",
    validate="one_to_one"
)

data["is_affected"] = data["is_affected"].fillna(False)

# Define clean development and reserved evaluation sets
features = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence"
]

train_data = data[
    (data["split"] == "internal_train") &
    (~data["is_affected"])
].copy()

test_data = data[
    (data["split"] == "reserved_final_test") &
    (~data["is_affected"])
].copy()

X_train = train_data[features]
y_train = (
    train_data["next_arr_delay"] -
    train_data["current_arr_delay"]
)

X_test = test_data[features]
y_test = test_data["next_arr_delay"]

# Train the selected delta model
model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.06,
    max_iter=100,
    max_leaf_nodes=31,
    min_samples_leaf=40,
    l2_regularization=2,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=12,
    random_state=42
)

start = time.time()
model.fit(X_train, y_train)
training_time = time.time() - start

# Predict next-station delay
predicted_delta = model.predict(X_test)
predictions = (
    test_data["current_arr_delay"].to_numpy() +
    predicted_delta
)

# Evaluate
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print("=" * 60)
print("RAILPULSE AI — CLEAN DATA SENSITIVITY EXPERIMENT")
print("=" * 60)
print("Original dataset rows:", len(data))
print("Affected rows identified:", int(data["is_affected"].sum()))
print("Clean training rows:", len(train_data))
print("Clean reserved evaluation rows:", len(test_data))
print("Training time:", round(training_time, 2), "seconds")
print("-" * 60)
print("MAE :", round(mae, 4))
print("MSE :", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))
print("=" * 60)

# Save experiment results
results = pd.DataFrame([{
    "experiment": "clean_data_sensitivity",
    "training_rows": len(train_data),
    "evaluation_rows": len(test_data),
    "excluded_affected_rows": int(data["is_affected"].sum()),
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "training_time_seconds": training_time
}])

results.to_csv(
    base / "outputs/tables/sensitivity_experiment_results.csv",
    index=False
)

print("\nResults saved successfully.")


RAILPULSE AI — CLEAN DATA SENSITIVITY EXPERIMENT
Original dataset rows: 1101443
Affected rows identified: 183
Clean training rows: 699293
Clean reserved evaluation rows: 230814
Training time: 22.15 seconds
------------------------------------------------------------
MAE : 9.5034
MSE : 847.3491
RMSE: 29.1093
R²  : 0.8627

Results saved successfully.
